# Scripted Higher-Level Analyses in FSL using FEAT derivatives

This notebook walks through a simple, novice-friendly workflow in **Neurodesk EDU**:

1. set up one example project folder and one Level-2 FEAT template  
2. download completed Level-1 `.feat` directories for one subject from OSF  
3. keep downloaded Level-1 inputs separate from notebook-generated Level-2 derivatives  
4. render and run a fixed-effects Level-2 FEAT model from a trust-game template  
5. make a first pass at viewing the higher-level result in **NiiVue**

A small but important design choice in this revised version is that we separate **inputs** from **derivatives**. That makes the notebook safer to rerun, and it avoids confusing FEAT when old `.gfeat` directories already exist.


**Author**: <div style="margin-top: 10px;">
    <a href="https://orcid.org/0000-0001-5754-9633" target="_blank" style="color: #0066cc;">
        <i class="fab fa-orcid"></i> David V. Smith
    </a>
</div>

**Date**: April 4, 2026  

**License**: <div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>

**Provenance:** This notebook adapts course materials and selected logic from the Fareri trust-game workflow code, but rewrites the steps so they can be followed directly in notebook cells. Full workflow repository: `https://github.com/tubric/fareri-2022-neuroimage`.

**Acknowledgments:** This notebook was generated with assistance from ChatGPT-5 across several iterations and then revised by the instructor. The instructor reviewed the final content and takes responsibility for it.


### Citation and Resources

#### Study-specific references
- **Fareri et al. (2022)**: Fareri, D. S., Hackett, K., Tepfer, L. J., Kelly, V., Henninger, N., Reeck, C., Giovannetti, T., & Smith, D. V. (2022). *Age-related differences in ventral striatal and default mode network function during reciprocated trust.* **NeuroImage, 256**, 119267. https://doi.org/10.1016/j.neuroimage.2022.119267
- **Smith et al. (2024)**: Smith, D. V., Ludwig, R. M., Dennison, J. B., Reeck, C., & Fareri, D. S. (2024). *An fMRI Dataset on Social Reward Processing and Decision Making in Younger and Older Adults.* **Scientific Data, 11**(1), 158. https://doi.org/10.1038/s41597-024-02931-y

#### Full workflow repository
- **Fareri trust-game workflow repo**: `https://github.com/tubric/fareri-2022-neuroimage`

#### Public archive for this notebook
- **Persistent OSF project DOI**: [10.17605/OSF.IO/KTXUG](https://doi.org/10.17605/OSF.IO/KTXUG)
- **OSF file page**: [https://osf.io/ktxug/files/5rmxd](https://osf.io/ktxug/files/5rmxd)

#### Tools included in this workflow
- **FSL / FEAT**: Jenkinson, M., Beckmann, C. F., Behrens, T. E. J., Woolrich, M. W., & Smith, S. M. (2012). FSL. *NeuroImage, 62*, 782–790.
- **NiiVue / ipyniivue** for interactive image viewing in Jupyter: `https://github.com/niivue/niivue` and `https://github.com/niivue/jupyter-notebooks`
- **Neurodesk EDU**: `https://edu.neurodesk.org/`


## Table of content
[1. Load software tools and import python libraries](#1.-Load-software-tools-and-import-python-libraries)  
[2. Project setup](#2.-Project-setup)  
[3. Get the Level-1 FEAT directories from OSF](#3.-Get-the-Level-1-FEAT-directories-from-OSF)  
[4. Check the run-level FEAT inputs](#4.-Check-the-run-level-FEAT-inputs)  
[5. Render the Level-2 FEAT template](#5.-Render-the-Level-2-FEAT-template)  
[6. Run higher-level FEAT](#6.-Run-higher-level-FEAT)  
[7. Review the output](#7.-Review-the-output)  
[8. Optional: try a quick NiiVue visualization](#8.-Optional:-try-a-quick-NiiVue-visualization)  
[9. Dependencies in Jupyter/Python](#9.-Dependencies-in-Jupyter/Python)


## 1. Load software tools and import python libraries

In [ ]:

import module
await module.load('fsl/6.0.7.16')
await module.list()


In [ ]:
%%capture
!pip install ipyniivue nibabel numpy osfclient


In [ ]:
from pathlib import Path
import os
import numpy as np
import nibabel as nib
from IPython.display import display, Markdown, Image

HOME = Path.home()
EXAMPLE_DIR = HOME / "trust_l2_example"

DOWNLOAD_DIR = EXAMPLE_DIR / "downloads"
INPUT_ROOT = EXAMPLE_DIR / "inputs" / "fsl"
DERIV_ROOT = EXAMPLE_DIR / "derivatives" / "fsl"
TEMPLATE_ROOT = EXAMPLE_DIR / "templates"

for d in [DOWNLOAD_DIR, INPUT_ROOT, DERIV_ROOT, TEMPLATE_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

sub = "104"
task = "trust"
model_type = "act"
sm = 6
nruns = 5

print("Example directory:", EXAMPLE_DIR)
print("Downloads:", DOWNLOAD_DIR)
print("Input root:", INPUT_ROOT)
print("Derivative root:", DERIV_ROOT)

## 2. Project setup

We will keep everything for this example in one folder directly under the home directory:

`~/trust_l2_example`

This notebook is designed to be rerun safely. To make that possible, we keep the downloaded archive, the copied Level-1 FEAT inputs, and the notebook-generated Level-2 derivatives in separate subfolders.

For this notebook, we only need one new template file: the **5-run Level-2 fixed-effects FEAT template**. Unlike the first-level notebook, this workflow assumes the run-level `.feat` directories already exist and have been shared as a tar archive on OSF.

We will use four subfolders with different roles:

- `~/trust_l2_example/templates` for the downloaded `.fsf` template  
- `~/trust_l2_example/downloads` for the raw OSF archive and its unpacked contents  
- `~/trust_l2_example/inputs/fsl/sub-104` for downloaded Level-1 `.feat` directories  
- `~/trust_l2_example/derivatives/fsl/sub-104` for the new Level-2 `.fsf` and `.gfeat` derivatives

In [ ]:
%%bash
set -e

EXAMPLE_DIR="$HOME/trust_l2_example"

mkdir -p "$EXAMPLE_DIR/templates"
mkdir -p "$EXAMPLE_DIR/downloads"
mkdir -p "$EXAMPLE_DIR/inputs/fsl/sub-104"
mkdir -p "$EXAMPLE_DIR/derivatives/fsl/sub-104"

# Copy the one higher-level FEAT template we need for this notebook.
curl -L \
  https://raw.githubusercontent.com/tubric/fareri-2022-neuroimage/main/templates/L2_task-trust_model-01_type-act_nruns-5.fsf \
  -o "$EXAMPLE_DIR/templates/L2_task-trust_model-01_type-act_nruns-5.fsf"

In [ ]:
!find ~/trust_l2_example/templates -maxdepth 1 -type f | sort
!find ~/trust_l2_example/inputs/fsl -maxdepth 2 -type d | sort
!find ~/trust_l2_example/derivatives/fsl -maxdepth 2 -type d | sort

The original `L2stats.sh` script is designed for batch processing across many participants and many model types. It also handles study-specific exceptions for participants with missing or bad runs.

Here we simplify that logic for one concrete teaching case: **sub-104**, **five runs**, and the **activation (`act`) model**. The only major prerequisite is that the five Level-1 `.feat` directories already exist and are stored in the location that the Level-2 template expects.

One additional simplification is worth emphasizing: the downloaded Level-1 `.feat` directories are treated as read-only **inputs**, while the rendered Level-2 template and `.gfeat` directory are written to a separate **output** folder. That makes rerunning the notebook much safer.


## 3. Download the example FEAT directories from OSF

For this notebook, assume the completed FEAT directories have already been shared through the public OSF project:

- Persistent project DOI: [10.17605/OSF.IO/KTXUG](https://doi.org/10.17605/OSF.IO/KTXUG)
- OSF project: [https://osf.io/ktxug/](https://osf.io/ktxug/)
- File page: [https://osf.io/ktxug/files/5rmxd](https://osf.io/ktxug/files/5rmxd)

The first command below lists the files visible in that OSF project. In this case, the archive path reported by `osf ls` is:

`osfstorage/fareri-2022-neuroimage_example-activation.tar.gz`

The cell is written so that it can be rerun safely. If the archive already exists locally, it will be reused rather than downloaded again. The unpacked archive contents and the copied subject folder will still be refreshed, which keeps the notebook reproducible while avoiding unnecessary downloads.

Because the archive is hosted on a public OSF project with a persistent DOI, it is reasonable to treat it as a stable teaching resource for this example workflow.


In [ ]:
%%bash
set -e

EXAMPLE_DIR="$HOME/trust_l2_example"
PROJECT="ktxug"
TARGET_SUB="104"

REMOTE_ARCHIVE="osfstorage/fareri-2022-neuroimage_example-activation.tar.gz"
LOCAL_ARCHIVE="$EXAMPLE_DIR/downloads/fareri-2022-neuroimage_example-activation.tar.gz"

DOWNLOAD_DIR="$EXAMPLE_DIR/downloads"
SUB_INPUT_DIR="$EXAMPLE_DIR/inputs/fsl/sub-${TARGET_SUB}"

mkdir -p "$DOWNLOAD_DIR"
mkdir -p "$EXAMPLE_DIR/inputs/fsl"

echo "Files visible in OSF project $PROJECT:"
osf -p "$PROJECT" ls

echo
if [ ! -f "$LOCAL_ARCHIVE" ]; then
    echo "Downloading archive from OSF ..."
    osf -p "$PROJECT" fetch "$REMOTE_ARCHIVE" "$LOCAL_ARCHIVE"
else
    echo "Archive already exists locally. Reusing it:"
    echo "  $LOCAL_ARCHIVE"
fi

echo
echo "Refreshing unpacked archive contents ..."
rm -rf "$DOWNLOAD_DIR/__forOSF"
tar -xzf "$LOCAL_ARCHIVE" -C "$DOWNLOAD_DIR"

echo
echo "Refreshing subject input directory ..."
rm -rf "$SUB_INPUT_DIR"
mkdir -p "$SUB_INPUT_DIR"

if [ -d "$DOWNLOAD_DIR/__forOSF/sub-${TARGET_SUB}" ]; then
    cp -r "$DOWNLOAD_DIR/__forOSF/sub-${TARGET_SUB}/." "$SUB_INPUT_DIR/"
else
    echo "Expected folder $DOWNLOAD_DIR/__forOSF/sub-${TARGET_SUB} was not found."
    echo "Check the archive contents and adjust the copy path in this cell if needed."
    exit 1
fi

echo
echo "Subject inputs are now here:"
echo "  $SUB_INPUT_DIR"
find "$SUB_INPUT_DIR" -maxdepth 1 -type d | sort



In [ ]:
!find ~/trust_l2_example/downloads/__forOSF -maxdepth 2 -type d | sort | head -40

A quick practical note: this archive is expected to unpack into a top-level `__forOSF/` folder. The cell above then copies only the participant folder needed for this notebook into:

`~/trust_l2_example/inputs/fsl/sub-104/`

That keeps the downloaded materials separate from the new Level-2 derivatives we are about to generate in `~/trust_l2_example/derivatives/`. It also means the notebook can be rerun without depending on where a user may have stored earlier analyses.


## 4. Check the run-level FEAT inputs

Level-2 FEAT combines a set of completed Level-1 `.feat` directories. Before doing anything else, confirm that those run-level inputs actually exist and follow the naming convention the template expects.

For this simple example, the expected inputs are:
- `L1_task-trust_model-01_type-act_run-01_sm-6.feat`
- `L1_task-trust_model-01_type-act_run-02_sm-6.feat`
- `L1_task-trust_model-01_type-act_run-03_sm-6.feat`
- `L1_task-trust_model-01_type-act_run-04_sm-6.feat`
- `L1_task-trust_model-01_type-act_run-05_sm-6.feat`

These checks are deliberately simple. If one or more runs are missing, it is better to stop here than to render an `.fsf` file that points to nonexistent inputs.


In [ ]:
%%bash
set -e

INPUT_DIR="$HOME/trust_l2_example/inputs/fsl/sub-104"

for run in 01 02 03 04 05; do
  featdir="$INPUT_DIR/L1_task-trust_model-01_type-act_run-${run}_sm-6.feat"
  if [ -d "$featdir" ]; then
    echo "FOUND: $featdir"
  else
    echo "MISSING: $featdir"
  fi
done

In [ ]:
!find ~/trust_l2_example/inputs/fsl/sub-104 -maxdepth 1 -type d -name "L1_task-trust_model-01_type-act_run-*_sm-6.feat" | sort


If one or more runs are missing, stop here and go back to the first-level workflow. The original `L2stats.sh` script has special-case logic for participants with missing runs, but it is better to keep the teaching example simple and fully explicit.


One practical point is still worth noting: these Level-1 FEAT directories were generated with **FSL 6.0.7.16**, and this notebook assumes the same version for consistency. In principle, higher-level FEAT often works across nearby FSL versions, but for a teaching example it is better to keep the software version fixed so the file layout and behavior stay predictable.


## 5. Render the Level-2 FEAT template

The original script uses `sed` to replace a small set of placeholders in the `.fsf` template. For the 5-run activation model, the important placeholders are:

- `OUTPUT` for the new higher-level output directory  
- `INPUT1` through `INPUT5` for the five run-level `.feat` directories

This is the key idea of scripted FEAT workflows: rather than point-and-clicking through the GUI every time, you render a text template with the correct paths and then hand the finished `.fsf` file to FEAT.

In this notebook, the rendered `.fsf` is written to the **output** folder rather than mixed in with the downloaded Level-1 inputs. That separation is what makes the rerun behavior much cleaner.


In [ ]:
%%bash
set -e

EXAMPLE_DIR="$HOME/trust_l2_example"
INPUT_DIR="$EXAMPLE_DIR/inputs/fsl/sub-104"
OUTPUT_DIR="$EXAMPLE_DIR/derivatives/fsl/sub-104"

ITEMPLATE="$EXAMPLE_DIR/templates/L2_task-trust_model-01_type-act_nruns-5.fsf"
OTEMPLATE="$OUTPUT_DIR/L2_task-trust_model-01_type-act.fsf"
OUTPUT="$OUTPUT_DIR/L2_task-trust_model-01_type-act_sm-6"

INPUT1="$INPUT_DIR/L1_task-trust_model-01_type-act_run-01_sm-6.feat"
INPUT2="$INPUT_DIR/L1_task-trust_model-01_type-act_run-02_sm-6.feat"
INPUT3="$INPUT_DIR/L1_task-trust_model-01_type-act_run-03_sm-6.feat"
INPUT4="$INPUT_DIR/L1_task-trust_model-01_type-act_run-04_sm-6.feat"
INPUT5="$INPUT_DIR/L1_task-trust_model-01_type-act_run-05_sm-6.feat"

sed \
  -e "s@OUTPUT@${OUTPUT}@g" \
  -e "s@INPUT1@${INPUT1}@g" \
  -e "s@INPUT2@${INPUT2}@g" \
  -e "s@INPUT3@${INPUT3}@g" \
  -e "s@INPUT4@${INPUT4}@g" \
  -e "s@INPUT5@${INPUT5}@g" \
  < "$ITEMPLATE" > "$OTEMPLATE"

echo "Rendered template: $OTEMPLATE"

In [ ]:
!grep -E 'outputdir|multiple|ncopeinputs|feat_files' ~/trust_l2_example/derivatives/fsl/sub-104/L2_task-trust_model-01_type-act.fsf | head -20

That quick check is worth doing. It lets you confirm that the rendered `.fsf` now points to the correct output directory and the correct five inputs before you launch FEAT.

It is especially helpful in a notebook, because once FEAT starts, any path mistake becomes much harder to debug from the long log output alone.


## 6. Run higher-level FEAT

The next two cells prepare and launch FEAT on the rendered Level-2 `.fsf` file. Compared with fMRIPrep, this step is usually much faster, but it can still take a while depending on the system and the size of the run-level inputs.

In conceptual terms, this model uses **fixed effects** to combine the five trust-task runs within one participant. The result is a single `.gfeat` directory that contains one higher-level FEAT result for each lower-level cope.

Before running FEAT, we will remove only the notebook-generated Level-2 output for this example. That prevents rerun collisions without touching the downloaded Level-1 `.feat` inputs.


### Clean up any previous Level-2 output for this notebook

If this notebook has been run before, FEAT may try to avoid an existing output directory by creating a new one with a `+` in the name. That is not usually a serious problem, but it can confuse later cells that expect one stable output path.

The cleanup cell below removes only the notebook-generated Level-2 output directory for this example. It leaves the downloaded Level-1 inputs untouched.


In [ ]:
%%bash
set -e

OUTPUT_DIR="$HOME/trust_l2_example/derivatives/fsl/sub-104"
L2_BASENAME="L2_task-trust_model-01_type-act_sm-6"

rm -rf "${OUTPUT_DIR}/${L2_BASENAME}.gfeat"
rm -rf "${OUTPUT_DIR}/${L2_BASENAME}+"*.gfeat
rm -rf "${OUTPUT_DIR}/logs"

echo "Removed any previous notebook-generated Level-2 derivatives:"
echo "  ${OUTPUT_DIR}/${L2_BASENAME}.gfeat"
echo "  ${OUTPUT_DIR}/${L2_BASENAME}+*.gfeat"

In [ ]:
%%bash
set -e

OTEMPLATE="$HOME/trust_l2_example/derivatives/fsl/sub-104/L2_task-trust_model-01_type-act.fsf"
feat "$OTEMPLATE"

## 7. Review the output

The Level-2 output should now live in a `.gfeat` directory under the notebook's **output** folder. At a minimum, you want to confirm that:

- the `.gfeat` directory exists  
- the expected `cope*.feat` subdirectories exist  
- the FEAT report and design image exist  
- one or two thresholded maps exist for the contrasts you care about

Remember that each `cope*.feat` directory corresponds to one lower-level cope combined across runs. For thresholded maps inside a higher-level FEAT directory, the path is usually:

`copeX.feat/thresh_zstat1.nii.gz`


In [ ]:
from pathlib import Path

outroot = EXAMPLE_DIR / "derivatives" / "fsl" / "sub-104"
matches = sorted(
    outroot.glob("L2_task-trust_model-01_type-act_sm-6*.gfeat"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not matches:
    raise FileNotFoundError("No matching .gfeat directory found.")

gfeat_dir = matches[0]
print("Using gfeat directory:", gfeat_dir)

for rel in [
    "report.html",
    "design.png",
    "cope1.feat",
    "cope10.feat",
    "cope18.feat",
    "cope10.feat/thresh_zstat1.nii.gz",
]:
    p = gfeat_dir / rel
    print(f"{p}: {p.exists()}")


In [ ]:
cope_dirs = sorted(gfeat_dir.glob("cope*.feat"))
for p in cope_dirs[:20]:
    print(p)


In [ ]:
design_png = gfeat_dir / "design.png"
if design_png.exists():
    display(Image(filename=str(design_png)))
else:
    print("Run FEAT first, then come back to this cell.")


A nice didactic feature of the design image is that it stays simple at Level 2. For this fixed-effects model, the design matrix is essentially a run-average model: one EV, one contrast, and one row per run.

That simplicity makes Level 2 a good place to connect the FEAT GUI to the underlying logic of higher-level modeling. You are not changing the psychological contrast here. You are combining the same contrast across runs within one participant.


## 8. Optional: try quick NiiVue visualizations

The first cell below overlays one higher-level thresholded map on FEAT's own mean functional background image, `bg_image.nii.gz`. That background is blurrier than a structural template, but it is still useful because FEAT writes it in the same space and resolution as the higher-level statistics.

Here we use `cope10.feat/thresh_zstat1.nii.gz`, which corresponds to the **reciprocate > defect** contrast combined across runs.

The exact cope numbering comes from the first-level model. The important higher-level idea is simpler: you are now looking at the same contrast after FEAT has combined evidence across all five runs within this participant. Because the notebook keeps derivatives in a dedicated `~/trust_l2_example/derivatives` folder, reruns should continue to point to one stable `.gfeat` path.


In [ ]:
from ipyniivue import NiiVue

bg_meanfunc = gfeat_dir / "bg_image.nii.gz"
zmap = gfeat_dir / "cope10.feat" / "thresh_zstat1.nii.gz"

print("Background exists:", bg_meanfunc.exists(), bg_meanfunc)
print("Contrast exists:", zmap.exists(), zmap)

if bg_meanfunc.exists() and zmap.exists():
    bg_data = nib.load(str(bg_meanfunc)).get_fdata()
    bg_nonzero = bg_data[bg_data > 0]
    lo = float(np.percentile(bg_nonzero, 2))
    hi = float(np.percentile(bg_nonzero, 98))

    nv = NiiVue()
    nv.load_volumes([
        {
            "path": str(bg_meanfunc),
            "name": "bg_image",
            "colormap": "gray",
            "opacity": 1.0,
            "cal_min": lo,
            "cal_max": hi,
        },
        {
            "path": str(zmap),
            "name": "reciprocate > defect across runs",
            "colormap": "red",
            "opacity": 0.75,
        },
    ])
    display(nv)
else:
    print("Run FEAT first, then come back to this cell.")


A second visualization uses an MNI T1 background. We show it here because the anatomical contrast is easier to interpret visually, even though the background and higher-level statistics may not share exactly the same native resolution.


In [ ]:
from ipyniivue import NiiVue

# Try a few common FSL locations for the MNI T1 background.
bg_candidates = [
    Path(os.environ.get("FSLDIR", "")) / "data" / "standard" / "MNI152_T1_2mm_brain.nii.gz" if os.environ.get("FSLDIR") else None,
    Path("/opt/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz"),
    Path("/opt/fsl-6.0.7.16/data/standard/MNI152_T1_2mm_brain.nii.gz"),
    Path("/usr/share/fsl/5.0/data/standard/MNI152_T1_2mm_brain.nii.gz"),
]

bg_path = None
for cand in bg_candidates:
    if cand is not None and cand.exists():
        bg_path = cand
        break

zmap = gfeat_dir / "cope10.feat" / "thresh_zstat1.nii.gz"

print("Background exists:", bg_path is not None, bg_path)
print("Contrast exists:", zmap.exists(), zmap)

if bg_path is not None and zmap.exists():
    bg_data = nib.load(str(bg_path)).get_fdata()
    bg_nonzero = bg_data[bg_data > 0]
    lo = float(np.percentile(bg_nonzero, 2))
    hi = float(np.percentile(bg_nonzero, 98))

    nv = NiiVue()
    nv.load_volumes([
        {
            "path": str(bg_path),
            "name": "MNI152_T1_2mm_brain",
            "colormap": "gray",
            "opacity": 1.0,
            "cal_min": lo,
            "cal_max": hi,
        },
        {
            "path": str(zmap),
            "name": "reciprocate > defect across runs",
            "colormap": "red",
            "opacity": 0.75,
        },
    ])
    display(nv)
else:
    print("Could not find the MNI T1 background or the thresholded map.")


If you want to generalize this notebook later, the main changes are straightforward:

- change the participant ID  
- change the number of runs and use the matching `nruns-*` template  
- preserve any missing-run logic from the original `L2stats.sh` script  
- point the visualization cell to the cope you actually want to inspect

That is where the original shell script still earns its keep: it automates those bookkeeping details across many participants.

Just keep the same separation between downloaded **inputs**, cached **downloads**, and notebook-generated **derivatives**. That one design choice makes the scripted workflow much easier to rerun and troubleshoot.


## 9. Dependencies in Jupyter/Python
- Using the package [watermark](https://github.com/rasbt/watermark) to document system environment and software versions used in this notebook, alongside the Neurodesktop version extracted from the `JUPYTER_IMAGE` environment variable.


In [ ]:

import os

%load_ext watermark

%watermark
%watermark --iversions

neurodesktop_version = os.environ.get('JUPYTER_IMAGE', 'unknown').split(':')[-1]
print(f"Neurodesktop version: {neurodesktop_version}")
